# List Comprehensions & Generators

## Learning Map

`List comprehension → iterable comprehension → nested comprehension → conditional comprehension → dict comprehension → generator expression → lazy transformation → custom generator → time-series filtering`

## 1. Write a basic list comprehension

**Concept:** A list comprehension creates a new list by transforming every element of an iterable.

In [ ]:
class PriceSeries:
    def __init__(self, ticker, prices):
        self.ticker, self.prices = ticker, prices

    def adjusted_prices(self, uplift):
        return [price * (1 + uplift) for price in self.prices]

series = PriceSeries("BHP", [41.70, 42.15, 41.92, 43.08, 42.76])
print(series.adjusted_prices(0.01))

## 2. List comprehension over iterables

**Concept:** A comprehension can consume any iterable, including a custom iterable class.

In [1]:
class Portfolio:
    def __init__(self, positions): self.positions = positions
    def __iter__(self): return iter(self.positions.items())

portfolio = Portfolio({"BHP": .18, "CBA": .27, "CSL": .20, "RIO": .20, "WES": .15})
weights = [weight for ticker, weight in portfolio]
large = [ticker for ticker, weight in portfolio if weight >= .20]
print(weights)
print(large)

[0.18, 0.27, 0.2, 0.2, 0.15]
['CBA', 'CSL', 'RIO']


## 3. Writing list comprehensions

**Concept:** The expression can calculate a business metric rather than simply copying the source value.

In [2]:
class TradeBook:
    def __init__(self, trades): self.trades = trades
    def notionals(self):
        return [t["quantity"] * t["price"] for t in self.trades]
    def buy_notionals(self):
        return [t["quantity"] * t["price"] for t in self.trades if t["side"] == "BUY"]

book = TradeBook([
    {"ticker":"BHP","side":"BUY","quantity":2400,"price":41.70},
    {"ticker":"CBA","side":"SELL","quantity":800,"price":178.25},
    {"ticker":"CSL","side":"BUY","quantity":350,"price":312.40},
    {"ticker":"RIO","side":"BUY","quantity":1200,"price":119.80},
])
print(book.notionals())
print(book.buy_notionals())

[100080.0, 142600.0, 109339.99999999999, 143760.0]
[100080.0, 109339.99999999999, 143760.0]


## 4. Nested list comprehensions

**Concept:** Nested comprehensions express multiple iteration levels and are useful for matrix or multi-day market data.

In [3]:
class MarketMatrix:
    def __init__(self, prices): self.prices = prices
    def flatten(self): return [price for day in self.prices for price in day]
    def returns(self):
        return [[round(row[i]/row[i-1]-1,4) for i in range(1,len(row))] for row in self.prices]

matrix = MarketMatrix([[41.70,42.15,41.92,43.08],[178.25,179.10,177.80,180.20],[312.40,315.10,313.70,318.20]])
print(matrix.flatten())
print(matrix.returns())

[41.7, 42.15, 41.92, 43.08, 178.25, 179.1, 177.8, 180.2, 312.4, 315.1, 313.7, 318.2]
[[0.0108, -0.0055, 0.0277], [0.0048, -0.0073, 0.0135], [0.0086, -0.0044, 0.0143]]


## 5. Advanced comprehensions

**Concept:** Advanced comprehensions can combine filtering, transformation, and conditional output in one expression.

In [4]:
class RiskScanner:
    def __init__(self, observations): self.observations = observations
    def report(self):
        return [{"ticker":r["ticker"], "volatility":r["volatility"],
                 "risk": "HIGH" if r["volatility"]>=.30 else "MEDIUM" if r["volatility"]>=.20 else "LOW"}
                for r in self.observations if r["volume"] >= 500_000]

scanner = RiskScanner([
    {"ticker":"BHP","volatility":.28,"volume":1842000},
    {"ticker":"CBA","volatility":.19,"volume":932100},
    {"ticker":"CSL","volatility":.31,"volume":521800},
    {"ticker":"RIO","volatility":.34,"volume":1205000},
])
for row in scanner.report(): print(row)

{'ticker': 'BHP', 'volatility': 0.28, 'risk': 'MEDIUM'}
{'ticker': 'CBA', 'volatility': 0.19, 'risk': 'LOW'}
{'ticker': 'CSL', 'volatility': 0.31, 'risk': 'HIGH'}
{'ticker': 'RIO', 'volatility': 0.34, 'risk': 'HIGH'}


## 6. Using conditionals in comprehensions (1)

**Concept:** A trailing `if` filters elements before they enter the output list.

In [ ]:
class MarketFilter:
    def __init__(self, records): self.records = records
    def high_volume(self, threshold):
        return [r["ticker"] for r in self.records if r["volume"] >= threshold]

records = [{"ticker":"BHP","volume":1842000},{"ticker":"CBA","volume":932100},{"ticker":"CSL","volume":521800},{"ticker":"WES","volume":310000}]
print(MarketFilter(records).high_volume(800_000))